<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/Experiment_2_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 2: Two-Step Lookahead MEDAL vs. AB-MEDAL on GSM8K — Evaluation (20 Canonical + 50 Extended Questions)

This notebook runs **Experiment 2 (E2)** on the GSM8K validation/evaluation pool with `LLaDA-8B-Instruct` using the **Two-Step Lookahead FIG Reward**:
$$
R_{\mathrm{FIG}}(s, s_i) = \frac{H(s) - H(g_i)}{\max(H(s), \epsilon)}
$$

### Dataset Split Organization:
1. **Canonical 20 Samples** (`sample_slice=(0, 20)`): The exact 20 evaluation questions from Experiment 1, preserving historical prefix ordering.
2. **Additional 50 Samples** (`sample_slice=(20, 70)`): 50 new validation questions sampled under the deterministic `sample_seed=1` order.
3. **Combined 70 Samples**: Full validation pool evaluation and paired statistical tests.


## 1. Verify GPU Hardware
Verify CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Total VRAM: {vram_gb:.2f} GiB')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Verify 2-step lookahead FIG reward computation, state caching, and batched parity before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_experiment_2.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E2 unit tests failed!')
print('All E2 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e2_eval')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e2_eval'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Shared Model Loading
Configure budgets [256, 512], depth 10, batch size 4, and load `LLaDA-8B-Instruct`.

In [ ]:
from dlm_project.tasks.gsm8k.experiment_2 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context, load_model
)

METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
BATCH_SIZE = 4
RESUME = True

# Load base context to initialize model identity
base_ctx = context(stage='evaluation', methods=METHODS, budgets=BUDGETS)
print('Loading LLaDA-8B-Instruct wrapper on GPU in bfloat16...')
wrapper = load_model(base_ctx, device='cuda')
print('Model successfully loaded.')


## 7. Execute Part 1: Canonical Evaluation Set (Original 20 Questions)
Evaluates the exact 20 evaluation questions from Experiment 1 (`sample_slice=(0, 20)`).

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_2 import run_e2_stage

ctx_20 = context(stage='evaluation', methods=METHODS, budgets=BUDGETS, sample_slice=(0, 20))
print(f'Starting Part 1 (Canonical 20 Questions)...')
t0 = perf_counter()
q_rows_20, agg_rows_20 = run_e2_stage(
    wrapper,
    ctx_20,
    OUTPUT_DIR / 'canonical_20',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')


## 8. Execute Part 2: Extended Validation Set (Additional 50 Questions)
Evaluates the additional 50 validation questions (`sample_slice=(20, 70)`).

In [ ]:
ctx_50 = context(stage='evaluation', methods=METHODS, budgets=BUDGETS, sample_slice=(20, 70))
print(f'Starting Part 2 (Additional 50 Questions)...')
t0 = perf_counter()
q_rows_50, agg_rows_50 = run_e2_stage(
    wrapper,
    ctx_50,
    OUTPUT_DIR / 'extended_50',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes.')


## 9. Results & Paired Comparisons (Canonical 20, Extended 50, and Combined 70)

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e2 import compute_paired_comparison

print('=== Part 1: Canonical 20 Questions Paired Comparison ===')
paired_20 = compute_paired_comparison(q_rows_20, budgets=BUDGETS)
if paired_20:
    display(pd.DataFrame(paired_20)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']])

print('=== Part 2: Additional 50 Questions Paired Comparison ===')
paired_50 = compute_paired_comparison(q_rows_50, budgets=BUDGETS)
if paired_50:
    display(pd.DataFrame(paired_50)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']])

print('=== Combined Pool (All 70 Questions) Paired Comparison ===')
combined_q_rows = q_rows_20 + q_rows_50
paired_combined = compute_paired_comparison(combined_q_rows, budgets=BUDGETS)
if paired_combined:
    display(pd.DataFrame(paired_combined)[['search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy', 'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p']])


## 10. Synchronize All Artifacts to Google Drive
Flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
